# 06 - Change Detection Baseline (Siamese U-Net ResNet34)

Kaggle notebook for training change detection on xBD dataset.
Uses Siamese U-Net with shared ResNet34 encoder and concat_absdiff feature fusion.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # Attach xBD dataset + processed_xbd dataset to this notebook
    PROCESSED_DIR = Path('/kaggle/input/processed-xbd/processed_xbd')  # Adjust dataset slug
    OUTPUT_DIR = Path('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')
    !pip install -q segmentation-models-pytorch albumentations
else:
    PROCESSED_DIR = Path('../data/processed/xbd')
    OUTPUT_DIR = Path('../outputs')
    sys.path.insert(0, str(Path('../src').resolve()))

print(f'Processed dir: {PROCESSED_DIR} (exists: {PROCESSED_DIR.exists()})')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
import torch
from torch.utils.data import DataLoader
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from rescuevision.data.datasets import XBDChangeDetectionDataset
from rescuevision.data.transforms import get_segmentation_transforms
from rescuevision.models.siamese_unet import SiameseUNet
from rescuevision.models.losses import BCEDiceLoss
from rescuevision.training.train_change_detection import train_change_detection
from rescuevision.utils.seed import set_seed

## Configuration

In [ ]:
config = {
    "seed": 42,
    "device": "cuda",
    "dataset": {
        "processed_dir": str(PROCESSED_DIR),
        "split_csv": str(PROCESSED_DIR / "metadata" / "splits_event_aware.csv"),
        "image_size": 512,
        "batch_size": 4,
        "num_workers": 2,
    },
    "model": {
        "encoder_name": "resnet34",
        "encoder_weights": "imagenet",
        "fusion_mode": "concat_absdiff",
        "classes": 1,
    },
    "training": {
        "epochs": 50,
        "lr": 1e-4,
        "weight_decay": 1e-5,
        "scheduler": "cosine",
        "mixed_precision": True,
        "early_stopping_patience": 8,
        "save_best_metric": "iou",
        "gradient_accumulation_steps": 1,
    },
    "loss": {
        "bce_weight": 0.5,
        "dice_weight": 0.5,
    },
    "augmentation": {
        "horizontal_flip": 0.5,
        "vertical_flip": 0.5,
        "rotate90": 0.5,
        "brightness_contrast": 0.2,
        "normalize": True,
    },
}

set_seed(config["seed"])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

## Create Datasets

In [ ]:
train_transform = get_segmentation_transforms(
    config["augmentation"], phase="train", image_size=512,
)
val_transform = get_segmentation_transforms(
    config["augmentation"], phase="val", image_size=512,
)

train_dataset = XBDChangeDetectionDataset(
    processed_dir=config["dataset"]["processed_dir"],
    split_csv=config["dataset"]["split_csv"],
    split="train",
    transform=train_transform,
    image_size=512,
)

val_dataset = XBDChangeDetectionDataset(
    processed_dir=config["dataset"]["processed_dir"],
    split_csv=config["dataset"]["split_csv"],
    split="val",
    transform=val_transform,
    image_size=512,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["dataset"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config["dataset"]["batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f'Train: {len(train_dataset)} tiles, Val: {len(val_dataset)} tiles')

## Create Model and Train

In [ ]:
model = SiameseUNet(
    encoder_name=config["model"]["encoder_name"],
    encoder_weights=config["model"]["encoder_weights"],
    fusion_mode=config["model"]["fusion_mode"],
    classes=config["model"]["classes"],
)

loss_fn = BCEDiceLoss(
    bce_weight=config["loss"]["bce_weight"],
    dice_weight=config["loss"]["dice_weight"],
)

output_dir = OUTPUT_DIR / "change_detection_siamese"

best_metrics = train_change_detection(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    config=config,
    output_dir=str(output_dir),
)

print(f'\nBest metrics: {best_metrics}')

## Visualize Predictions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model.eval()
model.to(device)

fig, axes = plt.subplots(4, 4, figsize=(20, 20))

for i in range(4):
    batch = next(iter(val_loader))
    pre_imgs = batch["pre_image"][:1].to(device)
    post_imgs = batch["post_image"][:1].to(device)
    masks_gt = batch["mask"][:1].numpy()
    
    with torch.no_grad():
        pred = torch.sigmoid(model(pre_imgs, post_imgs)).squeeze().cpu().numpy()
    pred_binary = (pred > 0.5).astype(np.uint8)
    
    # Denormalize pre image
    pre_img = pre_imgs[0].cpu().numpy().transpose(1, 2, 0)
    pre_img = (pre_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    pre_img = np.clip(pre_img, 0, 1)
    
    # Denormalize post image
    post_img = post_imgs[0].cpu().numpy().transpose(1, 2, 0)
    post_img = (post_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    post_img = np.clip(post_img, 0, 1)
    
    axes[0, i].imshow(pre_img)
    axes[0, i].set_title(f'Pre: {batch["tile_id"][0][:30]}')
    axes[1, i].imshow(post_img)
    axes[1, i].set_title('Post')
    axes[2, i].imshow(masks_gt[0], cmap='gray')
    axes[2, i].set_title('Ground Truth Change')
    axes[3, i].imshow(pred_binary, cmap='gray')
    axes[3, i].set_title('Predicted Change')
    
    for ax in axes[:, i]:
        ax.axis('off')

plt.tight_layout()
plt.savefig(output_dir / 'predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Change Overlay Visualization

In [ ]:
# Overlay predicted change mask on post-disaster image
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i in range(4):
    batch = next(iter(val_loader))
    pre_imgs = batch["pre_image"][:1].to(device)
    post_imgs = batch["post_image"][:1].to(device)
    masks_gt = batch["mask"][:1].numpy()
    
    with torch.no_grad():
        pred = torch.sigmoid(model(pre_imgs, post_imgs)).squeeze().cpu().numpy()
    pred_binary = (pred > 0.5).astype(np.uint8)
    
    # Denormalize post image
    post_img = post_imgs[0].cpu().numpy().transpose(1, 2, 0)
    post_img = (post_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
    post_img = np.clip(post_img, 0, 1)
    
    # Create overlay: red channel for GT change, green for predicted change
    overlay = post_img.copy()
    overlay_gt = post_img.copy()
    
    # GT overlay (red)
    change_gt = masks_gt[0] > 0
    overlay_gt[change_gt] = [1.0, 0.2, 0.2]  # Red tint for changed areas
    
    # Prediction overlay (red)
    change_pred = pred_binary > 0
    overlay[change_pred] = [1.0, 0.2, 0.2]  # Red tint for predicted changed areas
    
    axes[0, i].imshow(overlay_gt)
    axes[0, i].set_title(f'GT Overlay: {batch["tile_id"][0][:25]}', fontsize=9)
    axes[1, i].imshow(overlay)
    axes[1, i].set_title('Pred Overlay', fontsize=9)
    
    for ax in axes[:, i]:
        ax.axis('off')

plt.suptitle('Change Detection Overlays (Red = Changed)', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'change_overlays.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Checkpoint

In [ ]:
if IS_KAGGLE:
    print('Checkpoint saved at:', output_dir / 'checkpoints')
    print('Save this output as a Kaggle dataset for the inference notebook.')
    !ls -la {output_dir}/checkpoints/
else:
    print('Training complete. Checkpoint saved.')